# Week 3 — Responses API, structured outputs, and secure development

Use the project-scoped `/openai/v1/responses` path for new Foundry work. Treat model output as untrusted, validate it at the application boundary, and inventory the model, SDK, prompt, data, and tools used by a release.

In [ ]:
import importlib.util
import sys
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
labs = helpers.load_offline_labs(curriculum_root)
session.safe_summary()

In [ ]:
class GroundedAnswer(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    answer: str = Field(min_length=1, max_length=200)
    citations: tuple[str, ...] = Field(min_length=1, max_length=8)
    uncertainty: str | None = None


synthetic_candidate = {
    "answer": "The project endpoint scopes access to project capabilities.",
    "citations": ("FOUNDATIONS-001",),
    "uncertainty": None,
}
validated = GroundedAnswer.model_validate(synthetic_candidate)
validated.model_dump()

In [ ]:
structured_candidates = {
    "valid": synthetic_candidate,
    "missing-field": {"answer": "A project endpoint scopes access."},
    "extra-field": {
        **synthetic_candidate,
        "debug_payload": "must never cross the boundary",
    },
    "oversized": {
        "answer": "x" * 201,
        "citations": ("FOUNDATIONS-001",),
        "uncertainty": None,
    },
}
structured_results = labs.exercise_structured_outputs(
    GroundedAnswer, structured_candidates
)

In [ ]:
safe_validation_view = {
    result.case_id: {
        "accepted": result.accepted,
        "issue_codes": result.issue_codes,
    }
    for result in structured_results
}
assert safe_validation_view["valid"]["accepted"]
assert "missing" in safe_validation_view["missing-field"]["issue_codes"]
assert "extra_forbidden" in safe_validation_view["extra-field"]["issue_codes"]
assert "string_too_long" in safe_validation_view["oversized"]["issue_codes"]
safe_validation_view

In [ ]:
request_policy = {"timeout_seconds": 30, "max_attempts": 2}
structured_support = {
    "schema": "GroundedAnswer/v1",
    "offline_boundary_verified": True,
    "selected_deployment_verified": False,
}
release_inventory = {
    "logical_model": session.logical_model,
    "deployment": session.deployment,
    "api": "project-scoped /openai/v1/responses",
    "sdk_family": "OpenAI SDK through aai-core native client",
    "prompt_digest": "record-at-release",
    "dataset_version": "foundry-curriculum-eval-v1",
    "tool_schema_versions": [],
    "dependency_scan_complete": False,
    "output_schema": "GroundedAnswer/v1",
    "request_policy": request_policy,
    "structured_support": structured_support,
}
release_inventory

## Exit criteria

Exercise valid, missing-field, extra-field, and oversized outputs. Confirm structured-output support for the selected model and scenario rather than assuming it is universal. Add timeouts, bounded retries, safe rendering, and dependency provenance to the release evidence.